# StandUp4AI: 1000-Video Evaluation (Fresh)

No rclone. No subprocess. Pure pathlib on mounted Drive.
Run all cells top to bottom.

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

In [ ]:
# Cell 2: Imports + paths
import os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import librosa
import torch, torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
warnings.filterwarnings('ignore')

BASE       = "/content/drive/MyDrive/standup4ai"
AUDIO_DIRS = [f"{BASE}/audio", f"{BASE}/audio_1000"]
LABELS_DIR = f"{BASE}/labels"
PARTITION  = f"{BASE}/standup4ai_partition.csv"
OUT_DIR    = f"{BASE}/eval_1000"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Audio:   {AUDIO_DIRS}")
print(f"Labels:  {LABELS_DIR}")
print(f"Partition: {PARTITION}")
print(f"Output:  {OUT_DIR}")

In [ ]:
# Cell 3: Scan Drive with pathlib (NO rclone)
# --- audio files ---
audio_files = {}
for d in AUDIO_DIRS:
    p = Path(d)
    if not p.exists():
        print(f"[WARN] audio dir missing: {d}")
        continue
    for ext in ('*.m4a', '*.mp3', '*.wav'):
        for f in p.glob(ext):
            audio_files[f.stem] = str(f)
print(f"Audio files found: {len(audio_files)}")

# --- label files (recursive — finds train/val/test subfolders too) ---
label_files = {}
lp = Path(LABELS_DIR)
if not lp.exists():
    print(f"[WARN] labels dir missing: {LABELS_DIR}")
else:
    for f in lp.rglob('*.csv'):
        label_files[f.stem] = str(f)
print(f"Label files found: {len(label_files)}")

# --- partition ---
df = pd.read_csv(PARTITION)
val_ids   = set(df[df['part'] == 'val']['fn'])
train_ids = set(df[df['part'] == 'train']['fn'])
print(f"Partition — val: {len(val_ids)}, train: {len(train_ids)}")

# --- eval-ready ---
val_eval   = val_ids   & audio_files.keys() & label_files.keys()
train_eval = train_ids & audio_files.keys() & label_files.keys()
print(f"Eval-ready — val: {len(val_eval)}, train: {len(train_eval)}, total: {len(val_eval)+len(train_eval)}")
assert len(val_eval) + len(train_eval) > 0, "ZERO eval-ready videos — check paths above!"

In [ ]:
# Cell 4: Feature extractor (15-dim prosody)
def extract_features(path, sr=22050):
    y, sr = librosa.load(path, sr=sr, mono=True)
    if len(y) < 0.5 * sr:
        return None
    spec_cent  = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    spec_bw    = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    spec_roll  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    zcr        = np.mean(librosa.feature.zero_crossing_rate(y))
    flatness  = np.mean(librosa.feature.spectral_flatness(y=y))
    rms       = np.mean(librosa.feature.rms(y=y))
    try:
        f0 = librosa.pyin(y, fmin=50, fmax=300, sr=sr)[0]
        f0_c = f0[~np.isnan(f0)]
        if len(f0_c) > 0:
            f0_mean, f0_std, f0_min, f0_max = np.mean(f0_c), np.std(f0_c), np.min(f0_c), np.max(f0_c)
        else:
            f0_mean = f0_std = f0_min = f0_max = 0.0
    except:
        f0_mean = f0_std = f0_min = f0_max = 0.0
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=5), axis=1)
    return np.array([spec_cent, spec_bw, spec_roll, zcr, flatness, rms,
                      f0_mean, f0_std, f0_min, f0_max, *mfcc])

# smoke test
_vid = next(iter(val_eval if val_eval else train_eval))
_f = extract_features(audio_files[_vid])
assert _f is not None and _f.shape == (15,), f"smoke test FAILED: {_f}"
print(f"Smoke test OK — {_vid} → shape {_f.shape}")

In [ ]:
# Cell 5: Auto-find + load model (searches entire Drive)
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(15, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        return self.net(x)

DRIVE_ROOT = Path("/content/drive/MyDrive")
SEARCH = [
    "standup4ai/models/*.pt",
    "standup4ai/*/best_model.pt",
    "models/*.pt",
    "*/best_model.pt",
    "*/top200*.pt",
    "*prosody*.pt",
]

print("Searching for model on Drive …")
candidates = []
for pat in SEARCH:
    for pt in sorted(DRIVE_ROOT.rglob(pat)):
        candidates.append((str(pt), pt.stat().st_size / 1024))

seen, unique = set(), []
for p, sz in candidates:
    if p not in seen:
        seen.add(p); unique.append((p, sz))

for p, sz in unique[:10]:
    print(f"  {sz:.0f}KB  {p}")

if not unique:
    raise FileNotFoundError(
        "No .pt model found on Drive. Upload top200_prosody_model.pt "
        "from autonomous_laughter_prediction_essential/models/ "
        "to your Drive standup4ai/models/")

MODEL_PATH = unique[0][0]
print(f"\nUsing: {MODEL_PATH}")

model = Net()
sd = torch.load(MODEL_PATH, map_location='cpu')
model.load_state_dict(sd, strict=False)
model.eval()
print(f"Model loaded ✓")

In [ ]:
# Cell 6: Evaluate with checkpoint every 25 videos
CKPT_EVERY = 25
CKPT_FILE  = f"{OUT_DIR}/eval_checkpoint.json"

def get_label(vid):
    try:
        df = pd.read_csv(label_files[vid])
    except:
        return None
    col = [c for c in df.columns if 'label' in c.lower()][0]
    return int((df[col].astype(str) == 'risa').any())

# resume from checkpoint
records = []
done    = set()
if Path(CKPT_FILE).exists():
    records = json.loads(Path(CKPT_FILE).read_text())
    done = {r['vid'] for r in records}
    print(f"Resuming from checkpoint: {len(done)} already done")

all_videos = sorted(list(val_eval | train_eval))
todo = [v for v in all_videos if v not in done]
print(f"Evaluating {len(todo)} of {len(all_videos)} videos …")

for i, vid in enumerate(tqdm(todo)):
    lab = get_label(vid)
    if lab is None:
        continue
    feats = extract_features(audio_files[vid])
    if feats is None:
        continue
    with torch.no_grad():
        prob = model(torch.tensor(feats, dtype=torch.float32).unsqueeze(0)).item()
    records.append({
        'vid': vid, 'prob': prob, 'pred': int(prob > 0.5),
        'label': lab, 'split': 'val' if vid in val_ids else 'train'
    })
    if (i + 1) % CKPT_EVERY == 0:
        Path(CKPT_FILE).write_text(json.dumps(records))

Path(CKPT_FILE).write_text(json.dumps(records))

all_true = [r['label'] for r in records]
all_pred = [r['pred'] for r in records]
print(f"\n=== RESULTS ({len(records)} videos) ===")
print(f"F1:        {f1_score(all_true, all_pred):.4f}")
print(f"Precision:  {precision_score(all_true, all_pred):.4f}")
print(f"Recall:    {recall_score(all_true, all_pred):.4f}")
for sp in ('val', 'train'):
    t = [r['label'] for r in records if r['split'] == sp]
    p = [r['pred']  for r in records if r['split'] == sp]
    if t:
        print(f"{sp}: F1={f1_score(t,p):.4f}  n={len(t)}")

In [ ]:
# Cell 7: Save results
results = {
    'f1':           float(f1_score(all_true, all_pred)),
    'precision':     float(precision_score(all_true, all_pred)),
    'recall':        float(recall_score(all_true, all_pred)),
    'n_videos':      len(records),
    'n_val':         sum(r['split']=='val'   for r in records),
    'n_train':       sum(r['split']=='train' for r in records),
    'pos_rate':      float(np.mean(all_true)),
    'pred_pos_rate': float(np.mean(all_pred)),
}

rp = f"{OUT_DIR}/eval_results.json"
cp = f"{OUT_DIR}/per_video_predictions.csv"
with open(rp, 'w') as f:
    json.dump(results, f, indent=2)
pd.DataFrame(records).to_csv(cp, index=False)
print(f"Saved: {rp}")
print(f"Saved: {cp}")
print()
print(json.dumps(results, indent=2))